## Youtube Summarizer Agent

In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from youtube_transcript_api import YouTubeTranscriptApi

In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
model = ChatGroq(
    model = "openai/gpt-oss-120b",
)

In [4]:
class YoutubeState(TypedDict):
  url:str
  video_id:str
  transcript:str
  summary:str

In [5]:
def extract_video_id(url:str)->str:
  if "v" in url:
    return url.split("v")[1].split("&")[0]
  elif "youtu.be/" in url:
        return url.split("youtu.be/")[1].split("?")[0]

  return url

In [6]:
def fetch_transcript_node(state: YoutubeState) -> dict:
    video_id = extract_video_id(state["url"])

    ytt = YouTubeTranscriptApi()
    transcript = ytt.fetch(video_id, languages=['en'])

    # Use item.text instead of item['text']
    full_text = " ".join([item.text for item in transcript])

    return {"video_id": video_id, "transcript": full_text}

In [7]:
def summarize_node(state: YoutubeState) -> dict:
    llm = model

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert content summarizer. Provide a concise, highly structured summary with key takeaways from the transcript."),
        ("user", "Transcript:\n{transcript}")
    ])

    chain = prompt | llm
    response = chain.invoke({"transcript": state["transcript"]})

    return {"summary": response.content}

In [8]:
builder = StateGraph(YoutubeState)

# Add nodes
builder.add_node("fetch_transcript", fetch_transcript_node)
builder.add_node("summarize", summarize_node)

# Connect nodes
builder.add_edge(START, "fetch_transcript")
builder.add_edge("fetch_transcript", "summarize")
builder.add_edge("summarize", END)

# Compile graph
app = builder.compile()

In [10]:
app

KeyboardInterrupt: 

In [11]:
# 5. Run the Agent
if __name__ == "__main__":
    youtube_url = "https://youtu.be/IjFcr5r0nMs?si=O_7rfxVsNZNjR2jV"

    result = app.invoke({"url": youtube_url})

    print("\n--- SUMMARY ---\n")
    print(result["summary"])


--- SUMMARY ---

**Video Summary – Simulating a ROS‑2 Robot in Gazebo**

| Section | What’s Covered | Key Take‑aways |
|---------|----------------|----------------|
| **Goal** | Build a 3‑D simulated robot that can be driven in a virtual room. | By the end you have a Gazebo world with a coloured robot that responds to velocity commands. |
| **Prerequisites** | • URDF (or Xacro) created in the previous tutorial.<br>• Basic knowledge of ROS 2 and Gazebo.<br>• (Optional) Gazebo‑ROS integration tutorial. | Watch the earlier URDF tutorial first; install Gazebo if not already present. |
| **Step‑by‑step Setup** | 1. **Run Robot State Publisher (RSP)** with `use_sim_time:=true`.<br>2. **Launch Gazebo** using the ROS‑2 launch file `gazebo_ros/gazebo.launch.py`.<br>3. **Spawn the robot** with `ros2 run gazebo_ros spawn_entity.py` (topic `robot_description`, give an entity name). | • `use_sim_time` synchronises all nodes to Gazebo’s clock.<br>• The three commands can be wrapped in a single laun